# 64) FamilyWise Error Rate (FWER) ve Alpha Inflation
Şimdiye kadar hep **tek bir hipotez testi** yapıyorduk (tek metrik, tek karşılaştırma). Ama gerçek hayatta genelde **birden fazla** test aynı anda yapılır (örn: bir A/B testinde hem "açılma oranı" hem "tıklama oranı" hem "satın alma oranı" aynı anda test edilir). Bu, ciddi bir gizli riski beraberinde getirir: **Alpha Inflation (Alfa Şişmesi).**

## Sorun Nedir?
Her tek testte α=0.05 kullanıldığında, "yanlışlıkla H0'ı reddetme" (Tip I Hata) riski %5. Ama **birden fazla test yaptığında**, bu 
riskler **birikir/toplanır** testlerin **en az birinde** yanlışlıkla "anlamlı" bir sonuç bulma ihtimalin, tek testteki %5'ten çok daha yüksek olur.

## Somut Örnek
Diyelim aynı A/B testinde **5 farklı metriği** (açılma, tıklama, satın alma, sepete ekleme, sayfa süresi) ayrı ayrı test ettik, her birinde α=0.05 kullandık. Gerçekte **hiçbirinde** gerçek bir fark olmasa bile (hepsinde H0 doğru olsa bile), **en az bir tanesinde** yanlışlıkla "anlamlı" sonuç bulma olasılığımız: $$P(\text{en az bir yanlış pozitif}) = 1-(1-\alpha)^m$$

$m$ = test sayısı. 5 test için: $1-(1-0.05)^5 \approx 0.226$ yani **%22.6**! Tek testteki %5'lik riskten **4.5 kat daha yüksek.**

## Çözüm: Düzeltilmiş Anlamlılık Eşikleri
Bu riski kontrol etmek için, her tek testteki α'yı **küçültürüz** — en yaygın yöntem **Bonferroni Düzeltmesi:**$$\alpha_{düzeltilmiş} = \frac{\alpha}{m}$$

5 test için: $\alpha_{düzeltilmiş} = 0.05/5 = 0.01$ — yani her tek testte, artık p<0.01 aranır, p<0.05 yeterli olmaz.

## Bonferroni'nin Dezavantajı
Çok **katı/muhafazakar** bir yöntemdir — Tip I Hata riskini iyi kontrol eder ama Tip II Hata riskini artırır. Çok sayıda test varsa (örn: 20+), gerçek etkileri bulmak çok zorlaşır.

## Alternatif Yöntemler (İsimlerini Bilmen Yeterli)
- **Holm-Bonferroni:** Bonferroni'den biraz daha az katı, adım adım düzeltme
- **Benjamini-Hochberg (FDR - False Discovery Rate):** Modern veri biliminde daha sık tercih edilir, çok sayıda test varken (örn: binlerce gen testi) daha dengeli bir yaklaşım sunar

## Neden Senin Alanın İçin Kritik?
A/B testlerinde "birden fazla metriği aynı anda izlemek" çok yaygın bir pratik hatadır (buna "multiple testing problem" denir) — bunu bilmeden test yapmak, yanlışlıkla "başarılı" sonuçlar bulma riskini ciddi şekilde artırır. Bu, ANOVA'dan (Konu 65+) sonra tekrar karşımıza çıkacak (Post-Hoc testlerde, Konu 67).

## Python'da Kullanımı
```python
from statsmodels.stats.multitest import multipletests

p_degerleri = [0.01, 0.03, 0.04, 0.20, 0.002]
reddedildi, duzeltilmis_p, _, _ = multipletests(p_degerleri, alpha=0.05, method='bonferroni')
```

In [1]:
import pandas as pd

# 1. Kurguladığımız 5 farklı P-değeri ve buton isimleri
butonlar = ["Mavi Buton", "Yeşil Buton", "Kırmızı Buton", "Sarı Buton", "Turuncu Buton"]
p_degerleri = [0.002, 0.015, 0.042, 0.080, 0.350]

# 2. Başlangıçtaki kırmızı çizgimiz (Alpha) ve test sayısı
alpha_ilk = 0.05
test_sayisi = len(p_degerleri)

# 3. Bonferroni Düzeltmesi: Yeni ve sertleştirilmiş Alpha sınırı
# Formül: Eski Alpha / Test Sayısı
alpha_yeni = alpha_ilk / test_sayisi

print(f"🎯 Başlangıç Alpha Sınırı: {alpha_ilk}")
print(f"🛡️ Çoklu Test Sonrası Yeni Sert Sınır (0.05 / 5): {alpha_yeni}\n")

# 4. Sonuçları karşılaştırmak için bir tablo oluşturalım
sonuclar = []
for buton, p in zip(butonlar, p_degerleri):
    # Düzeltmeden önce anlamlı mı? (p < 0.05)
    onceki_durum = "ANLAMLI (Başarılı) ✅" if p < alpha_ilk else "Anlamsız (Şans) ❌"
    
    # Düzeltmeden sonra anlamlı mı? (p < 0.01)
    sonraki_durum = "ANLAMLI (Gerçek Zafer) 🏆" if p < alpha_yeni else "ELENDİ (Yalancı Kahraman) 🚨"
    
    sonuclar.append({
        "Test Edilen": buton,
        "P-Değeri": p,
        "Düzeltme Öncesi (<0.05)": onceki_durum,
        "Düzeltme Sonrası (<0.01)": sonraki_durum
    })

# Tabloyu ekrana güzelce yazdıralım
df = pd.DataFrame(sonuclar)
print(df.to_string(index=False))


🎯 Başlangıç Alpha Sınırı: 0.05
🛡️ Çoklu Test Sonrası Yeni Sert Sınır (0.05 / 5): 0.01

  Test Edilen  P-Değeri Düzeltme Öncesi (<0.05)    Düzeltme Sonrası (<0.01)
   Mavi Buton     0.002    ANLAMLI (Başarılı) ✅    ANLAMLI (Gerçek Zafer) 🏆
  Yeşil Buton     0.015    ANLAMLI (Başarılı) ✅ ELENDİ (Yalancı Kahraman) 🚨
Kırmızı Buton     0.042    ANLAMLI (Başarılı) ✅ ELENDİ (Yalancı Kahraman) 🚨
   Sarı Buton     0.080       Anlamsız (Şans) ❌ ELENDİ (Yalancı Kahraman) 🚨
Turuncu Buton     0.350       Anlamsız (Şans) ❌ ELENDİ (Yalancı Kahraman) 🚨


In [2]:
from statsmodels.stats.multitest import multipletests

p_degerleri = [0.002, 0.015, 0.042, 0.080, 0.350]
butonlar = ["Mavi Buton", "Yeşil Buton", "Kırmızı Buton", "Sarı Buton", "Turuncu Buton"]

reddedildi, duzeltilmis_p, _, _ = multipletests(p_degerleri, alpha=0.05, method='bonferroni')

for buton, p, red, dp in zip(butonlar, p_degerleri, reddedildi, duzeltilmis_p):
    print(f"{buton}: orijinal p={p}, düzeltilmiş p={dp:.4f}, H0 reddedildi mi? {red}")

Mavi Buton: orijinal p=0.002, düzeltilmiş p=0.0100, H0 reddedildi mi? True
Yeşil Buton: orijinal p=0.015, düzeltilmiş p=0.0750, H0 reddedildi mi? False
Kırmızı Buton: orijinal p=0.042, düzeltilmiş p=0.2100, H0 reddedildi mi? False
Sarı Buton: orijinal p=0.08, düzeltilmiş p=0.4000, H0 reddedildi mi? False
Turuncu Buton: orijinal p=0.35, düzeltilmiş p=1.0000, H0 reddedildi mi? False


### Sonuç
Normalde tek bir test yaptığımızda kendimize %5 yanılma payı koyuyorduk bir başka deyişe %95 doğru tahmin oranımız vardı ancak örneğin aynı anda 3 test yaptığımızda 0.95 x 0.95 x 0.95 = %85.7375 oranında doğruluk payımız varken en az bir testte tip 1 hata yapma ihtimaliz %14.2625'e fırladı, bu çok yüksek bir oran !!! işte bu yüksek orandan kaynaklanan yanlış sonuçları gidermek adına düzeltmeler yaparak alfa değerimizi küçülterek hangi testin gerçekten anlamlı olduğuna karar verebiliriz. Yukarıdaki örnekte alpha değerini bonferroni düzeltmesi uygulayarak %5'den %1'e çektik sonuç olarak sadece mavi butonun anlamlı bir farka sahip olduğu sonucuna vardık.